In [66]:
import pandas as pd
import numpy as np
import os
import yaml
import json

# Load config
config_path = "/app/bindmount/gedi_data_2/config.yaml"
with open(config_path, "r") as file:
        config = yaml.safe_load(file)

df_path = config["paths"]["results"]["filtering_fine"]
fine_score_df = pd.read_json(df_path)
fine_score_df

,763620,763621,763638,763640,763660
763620,"{'ransac_fitness': 1.0, 'ransac_inlier_rmse': ...","{'ransac_fitness': 0.9998, 'ransac_inlier_rmse...",None,None,None
763621,"{'ransac_fitness': 1.0, 'ransac_inlier_rmse': ...","{'ransac_fitness': 0.9998, 'ransac_inlier_rmse...",None,None,None
767340,"{'ransac_fitness': 1.0, 'ransac_inlier_rmse': ...","{'ransac_fitness': 0.9998, 'ransac_inlier_rmse...",None,None,None
777133,"{'ransac_fitness': 0.0, 'ransac_inlier_rmse': ...","{'ransac_fitness': 0.911, 'ransac_inlier_rmse'...",None,None,None
777239,"{'ransac_fitness': 0.7758, 'ransac_inlier_rmse...","{'ransac_fitness': 0.7222000000000001, 'ransac...",None,None,None
...,...,...,...,...,...
8092171,None,None,None,None,"{'ransac_fitness': 1.0, 'ransac_inlier_rmse': ..."
8092190,None,None,None,None,"{'ransac_fitness': 0.9938, 'ransac_inlier_rmse..."
8093118,None,None,None,None,"{'ransac_fitness': 0.9946, 'ransac_inlier_rmse..."
8141343,None,None,None,None,"{'ransac_fitness': 1.0, 'ransac_inlier_rmse': ..."


In [74]:
methods_ranking = {}

for scan in fine_score_df.columns:  # Iterate over rows (scans are the index)
    
    df = fine_score_df[scan].dropna()  # Drop NaN values (drop all the cad that were not candidates)
    df = pd.DataFrame(list(df))  # Convert Series of dictionaries to DataFrame
    df.index = fine_score_df.index[df.index]  # Set the index to the names of the candidates
    
    df_sorted_fitness = df.sort_values("ransac_fitness", ascending=False)
    df_sorted_rmse = df.sort_values("ransac_inlier_rmse", ascending=True)
    df_sorted_fit_rmse = df.sort_values(["ransac_fitness", "ransac_inlier_rmse"], ascending=[False, True])
    df_sorted_rmse_fit = df.sort_values(["ransac_inlier_rmse", "ransac_fitness"], ascending=[True, False])
    
    # Check if the scan exists in the index before attempting to get its location
    index_fitness = df_sorted_fitness.index.get_loc(scan) + 1 if scan in df_sorted_fitness.index else None
    index_rmse = df_sorted_rmse.index.get_loc(scan) + 1 if scan in df_sorted_rmse.index else None
    index_fit_rmse = df_sorted_fit_rmse.index.get_loc(scan) + 1 if scan in df_sorted_fit_rmse.index else None
    index_rmse_fit = df_sorted_rmse_fit.index.get_loc(scan) + 1 if scan in df_sorted_rmse_fit.index else None
    
    methods_ranking[scan] = {
        "fitness": index_fitness,
        "rmse": index_rmse,
        "fit_rmse": index_fit_rmse,
        "rmse_fit": index_rmse_fit
    }
    
 # Process only the first column for demonstration purposes
methods_ranking

{763620: {'fitness': 1, 'rmse': 3, 'fit_rmse': 1, 'rmse_fit': 3},
 763621: {'fitness': 2, 'rmse': 3, 'fit_rmse': 2, 'rmse_fit': 3},
 763638: {'fitness': 60, 'rmse': 58, 'fit_rmse': 60, 'rmse_fit': 58},
 763640: {'fitness': None, 'rmse': None, 'fit_rmse': None, 'rmse_fit': None},
 763660: {'fitness': None, 'rmse': None, 'fit_rmse': None, 'rmse_fit': None}}